# 05 — Transformer

Same task, same folds, same protocol as `04_lstm.ipynb`: predict battery capacity from
a 128-timestep charging window, with mileage withheld.

Five configurations run on fold 0, then the selected one across all 5 folds. Every run
logged to MLflow.

Method and results discussion: `docs/TRANSFORMER_METHOD.md`

## 1. Setup

In [ ]:
import json
import math
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import dagshub
import mlflow

RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

INDEX_DIR = Path("../data/processed")
CACHE_DIR = Path("../data/cache")
FIG_DIR = Path("../reports/figures")
REPORTS_DIR = Path("../reports")
for d in (CACHE_DIR, FIG_DIR, REPORTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

# Fold assignment fixed in notebook 03. Never regenerated - the baselines and the
# LSTM were scored on these exact folds, so changing them would void every comparison.
with open(INDEX_DIR / "fold_manifest.json") as fp:
    manifest = json.load(fp)

N_FOLDS = manifest["config"]["n_folds"]
car_to_fold = {int(k): v for k, v in manifest["car_to_fold"].items()}

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(f"vehicles {len(car_to_fold)}  folds {N_FOLDS}  device {device}")

In [ ]:
# Reference points, all out-of-sample on these same folds.

BASELINE_RMSE_MEAN = 1.877         # mean predictor, no input
BASELINE_RMSE_LINEAR = 1.095       # mileage-only linear regression
BASELINE_STD_LINEAR = 0.21
PAPER_LSTM_RMSE = 1.420            # reference paper, their fleet, single split

LSTM_RMSE = 1.727                  # notebook 04, final 5-fold
LSTM_STD = 0.045
LSTM_R2 = 0.101
LSTM_CORR = 0.395

BASELINE_FOLDS = [1.032, 1.081, 1.379, 0.815, 1.166]
LSTM_FOLDS = [1.726, 1.672, 1.717, 1.797, 1.725]

BASELINE_BANDS = {"(0, 50]": 0.777, "(50, 100]": 0.999, "(100, 150]": 0.968,
                  "(150, 200]": 1.409, "(200, 300]": 1.178}
LSTM_BANDS = {"(0, 50]": 2.600, "(50, 100]": 1.561, "(100, 150]": 1.235,
              "(150, 200]": 2.018, "(200, 300]": 1.980}

BANDS = [0, 50, 100, 150, 200, 300]

# Significance threshold, fixed before any Transformer result is seen.
# The LSTM's fold-to-fold std was 0.045, so a difference smaller than roughly
# twice that cannot be claimed as real, whichever direction it favours.
CLAIM_THRESHOLD = 2 * LSTM_STD

print(f"mean baseline   : {BASELINE_RMSE_MEAN}")
print(f"linear baseline : {BASELINE_RMSE_LINEAR} +/- {BASELINE_STD_LINEAR}")
print(f"LSTM            : {LSTM_RMSE} +/- {LSTM_STD}")
print(f"paper's LSTM    : {PAPER_LSTM_RMSE}")
print(f"\nclaim threshold : {CLAIM_THRESHOLD:.3f} RMSE")

## 2. Data

In [ ]:
# The 9-channel cache was built in notebook 04. Rebuilt here only if absent.

N_CHANNELS = 9
files = {k: CACHE_DIR / f"{k}_{N_CHANNELS}ch.npy" for k in ("X", "y", "cars", "mil")}

if all(p.exists() for p in files.values()):
    X, y, cars, mil = (np.load(files[k]) for k in ("X", "y", "cars", "mil"))
    print("Loaded 9-channel cache.")
else:
    paths, path_cars = [], []
    for car, car_files in manifest["files"].items():
        paths.extend(car_files)
        path_cars.extend([int(car)] * len(car_files))

    n = len(paths)
    X = np.empty((n, 128, 9), dtype=np.float32)
    y = np.empty(n, dtype=np.float32)
    mil = np.empty(n, dtype=np.float32)
    cars = np.asarray(path_cars, dtype=np.int32)

    for i, p in enumerate(tqdm(paths, desc="building cache")):
        arr, meta = torch.load(p, weights_only=False)
        raw = arr[:, :7]                                  # drop timestamp channel
        v_spread = (raw[:, 3] - raw[:, 4])[:, None]       # cell voltage imbalance
        t_spread = (raw[:, 5] - raw[:, 6])[:, None]       # temperature spread
        X[i] = np.hstack([raw, v_spread, t_spread])
        y[i] = meta["capacity"]
        mil[i] = meta["mileage"]                          # evaluation only

    for k, p in files.items():
        np.save(p, {"X": X, "y": y, "cars": cars, "mil": mil}[k])

folds = np.array([car_to_fold[int(v)] for v in cars], dtype=np.int8)

print(f"X        : {X.shape}  {X.nbytes / 1024**2:.0f} MB")
print(f"capacity : {y.min():.2f} - {y.max():.2f} Ah "
      f"(mean {y.mean():.2f}, std {y.std():.2f})")

In [ ]:
def split_indices(test_fold):
    """Three-way split by vehicle: 18 train / 6 validation / 6 test.

    Validation rotates with the test fold. The network needs held-out data to
    decide when to stop and which checkpoint to keep; using the test fold for
    those decisions would leak test information into training.
    """
    val_fold = (test_fold + 1) % N_FOLDS
    return (np.where((folds != test_fold) & (folds != val_fold))[0],
            np.where(folds == val_fold)[0],
            np.where(folds == test_fold)[0])


def fit_scaler(X_train):
    """Per-channel mean and std from the training split only."""
    mean = X_train.mean(axis=(0, 1), keepdims=True)
    std = X_train.std(axis=(0, 1), keepdims=True)
    std[std < 1e-8] = 1.0
    return mean, std


class SnippetDataset(Dataset):
    """Snippets held in memory, inputs and target both standardised."""

    def __init__(self, X, y, idx, mean, std, y_mean, y_std):
        self.X = ((X[idx] - mean) / std).astype(np.float32)
        self.y = ((y[idx] - y_mean) / y_std).astype(np.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        return torch.from_numpy(self.X[i]), torch.tensor(self.y[i])


def make_loaders(test_fold, batch_size=256):
    tr, va, te = split_indices(test_fold)
    mean, std = fit_scaler(X[tr])
    y_mean, y_std = float(y[tr].mean()), float(y[tr].std())
    kw = dict(mean=mean, std=std, y_mean=y_mean, y_std=y_std)

    loaders = {
        "train": DataLoader(SnippetDataset(X, y, tr, **kw),
                            batch_size=batch_size, shuffle=True),
        "val": DataLoader(SnippetDataset(X, y, va, **kw),
                          batch_size=batch_size, shuffle=False),
        "test": DataLoader(SnippetDataset(X, y, te, **kw),
                           batch_size=batch_size, shuffle=False),
    }
    return loaders, (y_mean, y_std), (tr, va, te)


for f in range(N_FOLDS):
    tr, va, te = split_indices(f)
    print(f"fold {f}:  train {len(tr):>6,}  val {len(va):>6,}  test {len(te):>6,}")

## 3. Model

In [ ]:
class PositionalEncoding(nn.Module):
    """Sinusoidal positional encoding, added to the embedded input.

    Self-attention is permutation-invariant: without positional information a
    Transformer sees the 128 timesteps as an unordered set. The encoding gives
    each position a distinct signature built from sine and cosine waves of
    geometrically increasing wavelength.

    Registered as a buffer, not a parameter - it is fixed, not learned.
    """

    def __init__(self, d_model, max_len=512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len, dtype=torch.float).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2).float()
                        * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))       # (1, max_len, d_model)

    def forward(self, x):
        return x + self.pe[:, : x.size(1)]

In [ ]:
class TransformerNet(nn.Module):
    """Transformer encoder over the charging window.

    Flow:
        (B, 128, 9)  input
        -> Linear    project 9 channels to d_model        (B, 128, d_model)
        -> + posenc  add positional information
        -> encoder   n_layers of self-attention + FFN     (B, 128, d_model)
        -> readout   collapse 128 positions to one vector (B, d_model)
        -> head      FC -> 1                              (B,)

    readout options:
        'last'  - final position, matching the LSTM's readout
        'mean'  - average over all 128 positions
        'cls'   - a learned summary token prepended to the sequence, which
                  attends to everything and carries the summary out

    use_posenc=False removes positional encoding entirely, making the model
    order-invariant. If performance is unaffected, the model is treating the
    window as an unordered bag of readings rather than a time series.
    """

    def __init__(self, n_features=9, d_model=64, n_heads=4, n_layers=2,
                 dim_ff=128, dropout=0.2, readout="last", use_posenc=True):
        super().__init__()
        self.readout = readout
        self.use_posenc = use_posenc

        self.input_proj = nn.Linear(n_features, d_model)
        self.posenc = PositionalEncoding(d_model) if use_posenc else None

        # A learned token prepended to the sequence. Self-attention lets it
        # gather from every position, so its output acts as a summary.
        if readout == "cls":
            self.cls = nn.Parameter(torch.zeros(1, 1, d_model))
            nn.init.normal_(self.cls, std=0.02)

        layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=dim_ff,
            dropout=dropout,
            batch_first=True,         # (batch, seq, feature)
            norm_first=True,          # pre-norm: more stable on small data
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=n_layers)

        self.head = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.LeakyReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 2, 1),
        )

        for m in self.head:
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        h = self.input_proj(x)                            # (B, 128, d_model)

        if self.readout == "cls":
            cls = self.cls.expand(h.size(0), -1, -1)
            h = torch.cat([cls, h], dim=1)                 # (B, 129, d_model)

        if self.posenc is not None:
            h = self.posenc(h)

        h = self.encoder(h)

        if self.readout == "mean":
            summary = h.mean(dim=1)
        elif self.readout == "cls":
            summary = h[:, 0, :]                           # the CLS position
        else:
            summary = h[:, -1, :]                          # final position

        return self.head(summary).squeeze(-1)

In [ ]:
def metrics(y_true, y_pred):
    """All metrics in Ah.

    corr and std_ratio distinguish a collapsed model (constant output) from a
    compressed one (tracks reality but hedges toward the mean).
    """
    return {
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "mape": float(np.mean(np.abs((y_true - y_pred) / y_true)) * 100),
        "r2": float(r2_score(y_true, y_pred)),
        "corr": float(np.corrcoef(y_pred, y_true)[0, 1]),
        "std_ratio": float(y_pred.std() / y_true.std()),
    }


def evaluate(model, loader, y_mean, y_std):
    """Predictions and targets, converted back to Ah."""
    model.eval()                                          # disable dropout
    preds, actuals = [], []

    with torch.no_grad():
        for xb, yb in loader:
            preds.append(model(xb.to(device)).cpu().numpy())
            actuals.append(yb.numpy())

    p = np.concatenate(preds) * y_std + y_mean
    a = np.concatenate(actuals) * y_std + y_mean
    return p, a

## 4. Training

In [ ]:
def train_fold(test_fold, cfg, verbose=False):
    """Train one fold. Returns (model, history, test_metrics, extras)."""
    loaders, (y_mean, y_std), (tr, va, te) = make_loaders(
        test_fold, cfg["batch_size"]
    )

    model = TransformerNet(
        n_features=cfg["n_features"], d_model=cfg["d_model"],
        n_heads=cfg["n_heads"], n_layers=cfg["n_layers"],
        dim_ff=cfg["dim_ff"], dropout=cfg["dropout"],
        readout=cfg["readout"], use_posenc=cfg["use_posenc"],
    ).to(device)

    criterion = nn.MSELoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg["lr"],
                                  weight_decay=cfg["weight_decay"])
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=4
    )

    history = {"train_loss": [], "val_rmse": [], "lr": []}
    best_val, best_state, stalled = float("inf"), None, 0

    for epoch in range(cfg["epochs"]):
        model.train()                                     # enable dropout
        running = 0.0

        for xb, yb in loaders["train"]:
            xb, yb = xb.to(device), yb.to(device)

            optimizer.zero_grad()                         # clear old gradients
            loss = criterion(model(xb), yb)               # forward + loss
            loss.backward()                               # gradients

            # Transformers are prone to occasional large gradients early in
            # training; clipping keeps updates bounded.
            nn.utils.clip_grad_norm_(model.parameters(), cfg["clip"])

            optimizer.step()                              # update parameters
            running += loss.item() * len(yb)

        train_loss = running / len(loaders["train"].dataset)

        vp, va_true = evaluate(model, loaders["val"], y_mean, y_std)
        val_rmse = float(np.sqrt(mean_squared_error(va_true, vp)))

        scheduler.step(val_rmse)
        history["train_loss"].append(train_loss)
        history["val_rmse"].append(val_rmse)
        history["lr"].append(optimizer.param_groups[0]["lr"])

        if val_rmse < best_val:                           # keep the best, not the last
            best_val = val_rmse
            best_state = {k: v.detach().clone()
                          for k, v in model.state_dict().items()}
            stalled = 0
        else:
            stalled += 1

        if verbose:
            print(f"  epoch {epoch+1:>3}  train {train_loss:.4f}  "
                  f"val_rmse {val_rmse:.4f}{'  *' if stalled == 0 else ''}")

        if stalled >= cfg["patience"]:
            break

    model.load_state_dict(best_state)
    tp, ta = evaluate(model, loaders["test"], y_mean, y_std)

    extras = {"test_idx": te, "test_pred": tp, "test_actual": ta,
              "epochs_run": len(history["train_loss"]), "best_val_rmse": best_val,
              "n_params": sum(p.numel() for p in model.parameters())}
    return model, history, metrics(ta, tp), extras


def band_metrics(extras):
    """Per-mileage-band metrics. Aggregate RMSE is dominated by the 100-150k
    band (41% of snippets), so a breakdown is needed to see where a model helps."""
    cut = pd.cut(mil[extras["test_idx"]] / 1000, bins=BANDS)
    rows = []
    for band in cut.categories:
        m = cut == band
        if m.sum() < 50:
            continue
        rows.append({"band": str(band), "n": int(m.sum()),
                     **metrics(extras["test_actual"][m], extras["test_pred"][m])})
    return rows

## 5. MLflow

In [ ]:
dagshub.init(repo_owner="RutikaKadam10",
             repo_name="ev-battery-capacity-prediction",
             mlflow=True)

mlflow.set_experiment("transformer")
print("tracking to:", mlflow.get_tracking_uri())

In [ ]:
def log_run(cfg, met, extras, history, bands=None, tags=None, checkpoint=None):
    """Log one training run to the active MLflow run."""
    mlflow.log_params(cfg)

    for ep, (tl, vr, lr_) in enumerate(zip(history["train_loss"],
                                           history["val_rmse"],
                                           history["lr"])):
        mlflow.log_metric("train_loss", tl, step=ep)
        mlflow.log_metric("val_rmse", vr, step=ep)
        mlflow.log_metric("learning_rate", lr_, step=ep)

    mlflow.log_metrics(met)
    mlflow.log_metric("epochs_run", extras["epochs_run"])
    mlflow.log_metric("best_val_rmse", extras["best_val_rmse"])
    mlflow.log_metric("n_params", extras["n_params"])

    mlflow.log_metric("gap_vs_mean_baseline", BASELINE_RMSE_MEAN - met["rmse"])
    mlflow.log_metric("gap_vs_linear_baseline", BASELINE_RMSE_LINEAR - met["rmse"])
    mlflow.log_metric("gap_vs_lstm", LSTM_RMSE - met["rmse"])

    if bands:
        for b in bands:
            key = (b["band"].replace("(", "").replace("]", "")
                   .replace(", ", "_").replace(" ", ""))
            mlflow.log_metric(f"rmse_band_{key}", b["rmse"])
            mlflow.log_metric(f"n_band_{key}", b["n"])

    if tags:
        mlflow.set_tags(tags)
    if checkpoint is not None:
        mlflow.log_artifact(str(checkpoint))

## 6. Ablation on fold 0

In [ ]:
BASE = {
    "model": "Transformer",
    "n_features": N_CHANNELS,
    "batch_size": 256,
    "lr": 3e-4,
    "weight_decay": 1e-4,
    "clip": 1.0,
    "epochs": 80,
    "patience": 10,
    "optimizer": "AdamW",
    "scheduler": "ReduceLROnPlateau",
    "norm_first": True,
    "seq_len": 128,
    "n_cars": len(car_to_fold),
    "n_folds": N_FOLDS,
    "seed": RANDOM_SEED,
    "timestamp_channel_dropped": True,
    "spread_features": True,
    "mileage_as_input": False,
}


def cfg(**kw):
    """Full config from BASE plus defaults plus overrides."""
    c = dict(BASE)
    c.update({"d_model": 64, "n_heads": 4, "n_layers": 2, "dim_ff": 128,
              "dropout": 0.2, "readout": "last", "use_posenc": True})
    c.update(kw)
    return c


ABLATIONS = [
    ("01-base",
     cfg(),
     {"change": "default encoder, last-position readout",
      "hypothesis": "matches the LSTM's readout for a like-for-like start"}),

    ("02-mean-pool",
     cfg(readout="mean"),
     {"change": "average all 128 positions instead of taking the last",
      "hypothesis": "unlike an RNN, no position is privileged in an encoder, "
                    "so the last position may be an arbitrary choice"}),

    ("03-cls-token",
     cfg(readout="cls"),
     {"change": "learned summary token prepended to the sequence",
      "hypothesis": "a dedicated token can learn what to gather rather than "
                    "weighting all positions equally"}),

    ("04-no-posenc",
     cfg(use_posenc=False),
     {"change": "positional encoding removed - model is order-invariant",
      "hypothesis": "if performance is unchanged, the model is not using "
                    "temporal order and treats the window as an unordered bag"}),

    ("05-small",
     cfg(d_model=32, n_heads=2, n_layers=1, dim_ff=64, dropout=0.3),
     {"change": "halved width, one layer, more dropout",
      "hypothesis": "the LSTM improved substantially when shrunk; 18 training "
                    "vehicles overfit a large model immediately"}),
]

for name, c, _ in ABLATIONS:
    print(f"{name:<16} d_model {c['d_model']:<4} heads {c['n_heads']}  "
          f"layers {c['n_layers']}  readout {c['readout']:<5} "
          f"posenc {c['use_posenc']}")

In [ ]:
ablation_results, ablation_histories = [], {}

with mlflow.start_run(run_name="transformer-ablation-fold0") as parent:
    mlflow.set_tags({"scope": "fold 0 only", "purpose": "configuration selection"})

    for name, c, notes in ABLATIONS:
        print(f"\n{'='*64}\n{name}\n{'='*64}")

        with mlflow.start_run(run_name=name, nested=True):
            t0 = time.time()
            model, hist, met, extras = train_fold(0, c, verbose=False)
            elapsed = time.time() - t0

            log_run(cfg={**c, "fold": 0}, met=met, extras=extras, history=hist,
                    bands=band_metrics(extras),
                    tags={**notes, "ablation": name})
            mlflow.log_metric("train_seconds", elapsed)

            ablation_results.append({"config": name, **met,
                                     "params": extras["n_params"],
                                     "epochs": extras["epochs_run"],
                                     "seconds": elapsed})
            ablation_histories[name] = hist

            print(f"  RMSE {met['rmse']:.4f}  R2 {met['r2']:>7.4f}  "
                  f"corr {met['corr']:.3f}  std_ratio {met['std_ratio']:.3f}  "
                  f"({elapsed:.0f}s, {extras['epochs_run']} ep, "
                  f"{extras['n_params']:,} params)")

    abl = pd.DataFrame(ablation_results)
    mlflow.log_metric("best_rmse", abl["rmse"].min())
    abl.to_csv(REPORTS_DIR / "transformer_ablation.csv", index=False)
    mlflow.log_artifact(str(REPORTS_DIR / "transformer_ablation.csv"))

print("\n" + "=" * 64)
print(abl.round(4).to_string(index=False))
print("=" * 64)
print(f"fold 0 — linear baseline {BASELINE_FOLDS[0]:.3f}  |  "
      f"LSTM {LSTM_FOLDS[0]:.3f}")

In [ ]:
# The positional-encoding test: compare 01-base against 04-no-posenc.
# Self-attention is permutation-invariant, so without positional encoding the
# model cannot distinguish one ordering of the 128 timesteps from another.

base_rmse = abl.loc[abl["config"] == "01-base", "rmse"].iloc[0]
nopos_rmse = abl.loc[abl["config"] == "04-no-posenc", "rmse"].iloc[0]
delta = nopos_rmse - base_rmse

print(f"01-base       RMSE {base_rmse:.4f}")
print(f"04-no-posenc  RMSE {nopos_rmse:.4f}")
print(f"difference    {delta:+.4f}")
print(f"threshold     {CLAIM_THRESHOLD:.4f}  (2x the LSTM's fold std)")
print()
if abs(delta) < CLAIM_THRESHOLD:
    print("Order does NOT appear to matter: removing positional encoding costs")
    print("less than the noise threshold. The model is effectively treating the")
    print("window as an unordered bag of 128 readings.")
else:
    print("Order matters: removing positional encoding degrades performance by")
    print("more than the noise threshold.")

In [ ]:
# Mark the selected configuration so the choice is visible in the run history.

best_name = abl.loc[abl["rmse"].idxmin(), "config"]
print(f"lowest RMSE on fold 0: {best_name}")

client = mlflow.tracking.MlflowClient()
runs = mlflow.search_runs(experiment_names=["transformer"],
                          filter_string="tags.ablation != ''")

for _, r in runs.iterrows():
    tag = r.get("tags.ablation")
    if isinstance(tag, str):
        client.set_tag(r["run_id"],
                       "status", "selected" if tag == best_name else "superseded")

print("status tags written")

In [ ]:
# Ablation comparison

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
names = abl["config"].tolist()
xpos = np.arange(len(names))

axes[0].bar(xpos, abl["rmse"], color="mediumpurple")
axes[0].axhline(LSTM_FOLDS[0], color="seagreen", ls="--", label="LSTM (fold 0)")
axes[0].axhline(BASELINE_FOLDS[0], color="darkorange", ls="--",
                label="linear baseline")
axes[0].set_xticks(xpos); axes[0].set_xticklabels(names, rotation=30, ha="right")
axes[0].set_ylabel("RMSE (Ah)"); axes[0].set_title("Test RMSE, fold 0")
axes[0].legend(fontsize=8)

axes[1].bar(xpos, abl["corr"], color="seagreen")
axes[1].axhline(LSTM_CORR, color="grey", ls=":", label="LSTM mean corr")
axes[1].set_xticks(xpos); axes[1].set_xticklabels(names, rotation=30, ha="right")
axes[1].set_ylabel("corr(pred, actual)"); axes[1].set_title("Signal captured")
axes[1].legend(fontsize=8)

axes[2].bar(xpos, abl["std_ratio"], color="indianred")
axes[2].axhline(1.0, color="black", ls=":", lw=1, label="matches reality")
axes[2].set_xticks(xpos); axes[2].set_xticklabels(names, rotation=30, ha="right")
axes[2].set_ylabel("pred std / actual std"); axes[2].set_title("Commitment")
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.savefig(FIG_DIR / "transformer_ablation.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Loss curves, all configurations on shared axes

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

for name, h in ablation_histories.items():
    axes[0].plot(h["train_loss"], label=name)
    axes[1].plot(h["val_rmse"], label=name)

axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Train MSE (scaled units)")
axes[0].set_title("Training loss"); axes[0].legend(fontsize=8)

axes[1].axhline(LSTM_FOLDS[0], color="seagreen", ls="--", lw=1, label="LSTM")
axes[1].axhline(BASELINE_FOLDS[0], color="darkorange", ls="--", lw=1,
                label="linear baseline")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Validation RMSE (Ah)")
axes[1].set_title("Validation RMSE"); axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig(FIG_DIR / "transformer_ablation_curves.png", dpi=150,
            bbox_inches="tight")
plt.show()

## 7. Selected configuration, full 5-fold

In [ ]:
# Selected on fold 0 validation performance. Edit here if the ablation favours
# a different configuration.
FINAL = dict(ABLATIONS[[n for n, _, _ in ABLATIONS].index(best_name)][1])
print(f"final configuration: {best_name}")
for k in ("d_model", "n_heads", "n_layers", "dim_ff", "dropout",
          "readout", "use_posenc", "lr"):
    print(f"  {k:<12} {FINAL[k]}")

In [ ]:
fold_results, band_rows, fold_histories = [], [], {}

with mlflow.start_run(run_name="transformer-final-5fold") as parent:
    mlflow.log_params(FINAL)
    mlflow.set_tags({"scope": "all folds", "status": "final",
                     "selected_config": best_name})

    for fold in range(N_FOLDS):
        print(f"\n{'='*64}\nFOLD {fold}\n{'='*64}")

        with mlflow.start_run(run_name=f"fold-{fold}", nested=True):
            t0 = time.time()
            model, hist, met, extras = train_fold(fold, FINAL, verbose=False)
            elapsed = time.time() - t0

            ckpt = REPORTS_DIR / f"transformer_fold{fold}.pt"
            torch.save(model.state_dict(), ckpt)

            bands = band_metrics(extras)
            log_run(cfg={**FINAL, "fold": fold}, met=met, extras=extras,
                    history=hist, bands=bands, checkpoint=ckpt)
            mlflow.log_metric("train_seconds", elapsed)

            for b in bands:
                band_rows.append({"fold": fold, **b})

            fold_results.append({"fold": fold, **met,
                                 "epochs": extras["epochs_run"],
                                 "seconds": elapsed})
            fold_histories[fold] = hist

            print(f"  RMSE {met['rmse']:.4f}  R2 {met['r2']:>7.4f}  "
                  f"corr {met['corr']:.3f}  ({elapsed:.0f}s, "
                  f"{extras['epochs_run']} epochs)")

    res = pd.DataFrame(fold_results)

    for m in ("rmse", "mae", "mape", "r2", "corr", "std_ratio"):
        mlflow.log_metric(f"{m}_mean", res[m].mean())
        mlflow.log_metric(f"{m}_std", res[m].std())

    gap_mean = BASELINE_RMSE_MEAN - res["rmse"].mean()
    gap_lstm = LSTM_RMSE - res["rmse"].mean()
    mlflow.log_metric("gap_vs_mean_baseline", gap_mean)
    mlflow.log_metric("gap_to_noise_ratio", gap_mean / res["rmse"].std())
    mlflow.log_metric("gap_vs_lstm", gap_lstm)

    res.to_csv(REPORTS_DIR / "transformer_per_fold.csv", index=False)
    bands_df = pd.DataFrame(band_rows)
    bands_df.to_csv(REPORTS_DIR / "transformer_per_band.csv", index=False)
    mlflow.log_artifact(str(REPORTS_DIR / "transformer_per_fold.csv"))
    mlflow.log_artifact(str(REPORTS_DIR / "transformer_per_band.csv"))

print("\n" + "=" * 64)
print(res.round(4).to_string(index=False))
print("=" * 64)
print(f"Transformer     : RMSE {res['rmse'].mean():.3f} ± {res['rmse'].std():.3f}")
print(f"LSTM            : RMSE {LSTM_RMSE} ± {LSTM_STD}")
print(f"linear baseline : RMSE {BASELINE_RMSE_LINEAR} ± {BASELINE_STD_LINEAR}")
print(f"mean baseline   : RMSE {BASELINE_RMSE_MEAN}")
print(f"paper's LSTM    : RMSE {PAPER_LSTM_RMSE}")

In [ ]:
# The headline comparison. The threshold was fixed before any Transformer
# result was seen, so this verdict is not chosen after the fact.

diff = LSTM_RMSE - res["rmse"].mean()

print(f"Transformer : {res['rmse'].mean():.3f} ± {res['rmse'].std():.3f}")
print(f"LSTM        : {LSTM_RMSE:.3f} ± {LSTM_STD:.3f}")
print(f"difference  : {diff:+.3f}  (positive = Transformer better)")
print(f"threshold   : {CLAIM_THRESHOLD:.3f}")
print()

if abs(diff) < CLAIM_THRESHOLD:
    print("NO CLAIMABLE DIFFERENCE. The gap is inside evaluation noise;")
    print("the two architectures are indistinguishable on this task.")
elif diff > 0:
    print(f"Transformer better by {diff:.3f}, which exceeds the threshold.")
else:
    print(f"LSTM better by {-diff:.3f}, which exceeds the threshold.")

print(f"\ngap over mean baseline : {BASELINE_RMSE_MEAN - res['rmse'].mean():.3f}")
print(f"gap / fold std         : "
      f"{(BASELINE_RMSE_MEAN - res['rmse'].mean()) / res['rmse'].std():.1f}x")

In [ ]:
# Per-band comparison against both the linear baseline and the LSTM

band_agg = bands_df.groupby("band").agg(
    folds=("fold", "nunique"),
    n_total=("n", "sum"),
    transformer_rmse=("rmse", "mean"),
    transformer_std=("rmse", "std"),
).round(3)

band_agg["lstm_rmse"] = [LSTM_BANDS.get(b, np.nan) for b in band_agg.index]
band_agg["baseline_rmse"] = [BASELINE_BANDS.get(b, np.nan) for b in band_agg.index]
band_agg["vs_lstm"] = (band_agg["transformer_rmse"]
                       - band_agg["lstm_rmse"]).round(3)

band_agg = band_agg.reindex(["(0, 50]", "(50, 100]", "(100, 150]",
                            "(150, 200]", "(200, 300]"])
band_agg

In [ ]:
# Three-way comparison plot

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))

axes[0].plot(res["fold"], res["rmse"], marker="o", color="mediumpurple",
             label="Transformer")
axes[0].plot(range(N_FOLDS), LSTM_FOLDS, marker="D", color="seagreen",
             label="LSTM")
axes[0].plot(range(N_FOLDS), BASELINE_FOLDS, marker="s", color="darkorange",
             label="Linear (mileage)")
axes[0].axhline(PAPER_LSTM_RMSE, color="crimson", ls="--", lw=1,
                label="Reference paper LSTM")
axes[0].axhline(BASELINE_RMSE_MEAN, color="grey", ls=":", lw=1,
                label="Mean baseline")
axes[0].set_xlabel("Fold"); axes[0].set_ylabel("RMSE (Ah)")
axes[0].set_xticks(range(N_FOLDS))
axes[0].set_title("By fold"); axes[0].legend(fontsize=8)

w = 0.27
idx = np.arange(len(band_agg))
axes[1].bar(idx - w, band_agg["baseline_rmse"], w, label="Linear (mileage)",
            color="darkorange")
axes[1].bar(idx, band_agg["lstm_rmse"], w, label="LSTM", color="seagreen")
axes[1].bar(idx + w, band_agg["transformer_rmse"], w, label="Transformer",
            color="mediumpurple")
axes[1].set_xticks(idx)
axes[1].set_xticklabels(band_agg.index, rotation=20, ha="right")
axes[1].set_xlabel("Mileage band (thousand km)"); axes[1].set_ylabel("RMSE (Ah)")
axes[1].set_title("By mileage band"); axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig(FIG_DIR / "all_models_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Diagnostics, and per-fold ranking against the other two models.
# If the models rank folds differently they are using different information.

print(res[["fold", "rmse", "r2", "corr", "std_ratio"]].round(3)
      .to_string(index=False))
print(f"\nmean corr      : {res['corr'].mean():.3f}  (LSTM: {LSTM_CORR})")
print(f"mean std_ratio : {res['std_ratio'].mean():.3f}")

rank = pd.DataFrame({
    "fold": range(N_FOLDS),
    "baseline": BASELINE_FOLDS,
    "lstm": LSTM_FOLDS,
    "transformer": res["rmse"].round(3),
})
print("\nper-fold RMSE, all three models:")
print(rank.round(3).to_string(index=False))
print("\nbest fold per model:")
for c in ("baseline", "lstm", "transformer"):
    print(f"  {c:<12} fold {int(rank[c].idxmin())}  "
          f"worst fold {int(rank[c].idxmax())}")

In [ ]:
# Final summary across every model in the project

summary = pd.DataFrame([
    {"model": "Mean baseline", "inputs": "none",
     "rmse": BASELINE_RMSE_MEAN, "std": 0.13, "r2": -0.052},
    {"model": "Linear regression", "inputs": "mileage",
     "rmse": BASELINE_RMSE_LINEAR, "std": BASELINE_STD_LINEAR, "r2": 0.635},
    {"model": "LSTM", "inputs": "128x9 sequence",
     "rmse": LSTM_RMSE, "std": LSTM_STD, "r2": LSTM_R2},
    {"model": "Transformer", "inputs": "128x9 sequence",
     "rmse": round(res["rmse"].mean(), 3), "std": round(res["rmse"].std(), 3),
     "r2": round(res["r2"].mean(), 3)},
    {"model": "Reference paper LSTM", "inputs": "128x8 sequence",
     "rmse": PAPER_LSTM_RMSE, "std": np.nan, "r2": np.nan},
])

summary.to_csv(REPORTS_DIR / "all_models_summary.csv", index=False)
summary